In [ ]:
# ==========================================================
# Install Required Packages
# ==========================================================

!pip -q install snntorch
!pip -q install dv
!pip -q install pandas
!pip -q install matplotlib
!pip -q install tqdm
!pip -q install scikit-learn

In [ ]:
# ==========================================================
# Import Libraries
# ==========================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import snntorch as snn
from snntorch import surrogate

from sklearn.model_selection import train_test_split

print("="*60)
print("PyTorch :", torch.__version__)
print("SNNTorch:", snn.__version__)
print("NumPy   :", np.__version__)
print("="*60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

PyTorch : 2.11.0+cpu
SNNTorch: 1.0.0
NumPy   : 2.0.2
Using device: cpu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================================
# Dataset Path
# ==========================================================

DATASET_PATH = "/content/drive/MyDrive/DVS128_Gesture/DvsGesture"

print(DATASET_PATH)
print(os.path.exists(DATASET_PATH))

/content/drive/MyDrive/DVS128_Gesture/DvsGesture
True


In [ ]:
# ==========================================================
# Check Dataset
# ==========================================================

files = sorted(os.listdir(DATASET_PATH))

print("Total files :", len(files))
print()

for f in files[:25]:
    print(f)

Total files : 250

LICENSE.txt
README.txt
errata.txt
gesture_mapping.csv
trials_to_test.txt
trials_to_train.txt
user01_fluorescent.aedat
user01_fluorescent_labels.csv
user01_fluorescent_led.aedat
user01_fluorescent_led_labels.csv
user01_lab.aedat
user01_lab_labels.csv
user01_led.aedat
user01_led_labels.csv
user01_natural.aedat
user01_natural_labels.csv
user02_fluorescent.aedat
user02_fluorescent_labels.csv
user02_fluorescent_led.aedat
user02_fluorescent_led_labels.csv
user02_lab.aedat
user02_lab_labels.csv
user02_led.aedat
user02_led_labels.csv
user02_natural.aedat


In [ ]:
aedat_files = sorted(
    [f for f in files if f.endswith(".aedat")]
)

label_files = sorted(
    [f for f in files if f.endswith("_labels.csv")]
)

print("AEDAT files :", len(aedat_files))
print("Label files :", len(label_files))

AEDAT files : 122
Label files : 122


In [ ]:
sample_csv = os.path.join(DATASET_PATH, label_files[0])

labels = pd.read_csv(sample_csv)

labels.head()

,class,startTime_usec,endTime_usec
0,1,80048239,85092709
1,2,89431170,95231007
2,3,95938861,103200075
3,4,114845417,123499505
4,5,124344363,131742581


In [ ]:
gesture_names = {
    1: "hand_clapping",
    2: "right_hand_wave",
    3: "left_hand_wave",
    4: "right_hand_clockwise",
    5: "right_hand_counter_clockwise",
    6: "left_hand_clockwise",
    7: "left_hand_counter_clockwise",
    8: "forearm_roll",
    9: "drums",
    10: "guitar",
    11: "other_gesture"
}

In [ ]:
import numpy as np

# Read one AEDAT file
sample_file = DATASET_PATH + "/" + aedat_files[0]

with open(sample_file, "rb") as f:

    # Skip header
    while True:
        line = f.readline()
        if line.startswith(b"#!END-HEADER"):
            break

    events = []

    while True:

        header = f.read(28)

        if len(header) < 28:
            break

        event_type = int.from_bytes(header[0:2], "little")
        event_size = int.from_bytes(header[4:8], "little")
        event_count = int.from_bytes(header[20:24], "little")

        data = f.read(event_size * event_count)

        if event_type != 1:
            continue

        packet = np.frombuffer(data, dtype=np.uint32)
        packet = packet.reshape(-1, 2)

        events.append(packet)

events = np.concatenate(events, axis=0)

print("Total events:", len(events))

Total events: 7790918


In [ ]:
addresses = events[:,0]
timestamps = events[:,1]

# Correct DVS128 decoding
x = (addresses >> 17) & 0x7F
y = (addresses >> 2) & 0x7F
p = (addresses >> 1) & 0x01

print("x range :", x.min(), x.max())
print("y range :", y.min(), y.max())
print("polarity:", np.unique(p))
print("timestamps:", timestamps.min(), timestamps.max())

x range : 0 127
y range : 0 127
polarity: [0 1]
timestamps: 80046394 194216418


In [ ]:
def extract_gesture(events_x,
                    events_y,
                    events_p,
                    events_t,
                    start_time,
                    end_time):
    """
    Extract all events within a given time interval.

    Parameters:
        events_x, events_y, events_p, events_t : NumPy arrays
            Full event stream.

        start_time : int
            Gesture start timestamp.

        end_time : int
            Gesture end timestamp.

    Returns:
        gx, gy, gp, gt : NumPy arrays
            Events belonging only to the selected gesture.
    """

    mask = (
        (events_t >= start_time) &
        (events_t <= end_time)
    )

    return (
        events_x[mask],
        events_y[mask],
        events_p[mask],
        events_t[mask]
    )

In [ ]:
NUM_BINS = 30
HEIGHT = 128
WIDTH = 128

def events_to_voxel(x, y, p, t):

    voxel = np.zeros(
        (NUM_BINS, 2, HEIGHT, WIDTH),
        dtype=np.float32
    )

    if len(t) == 0:
        return voxel

    t0 = t[0]
    t1 = t[-1]

    if t1 == t0:
        return voxel

    print("t dtype:", t.dtype)
    print("t min :", t.min())
    print("t max :", t.max())
    print("t0:", t0)
    print("t1:", t1)

    time_bins = (
        ((t - t0) / (t1 - t0)) * (NUM_BINS - 1)
    ).astype(np.int32)

    print("time_bins min:", time_bins.min())
    print("time_bins max:", time_bins.max())

    for i in range(len(x)):
        voxel[
            time_bins[i],
            p[i],
            y[i],
            x[i]
        ] += 1

    max_value = voxel.max()

    if max_value > 0:
        voxel /= max_value

    return voxel

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
train_trials = []

with open(os.path.join(DATASET_PATH, "trials_to_train.txt")) as f:
    for line in f:
        line = line.strip()
        if line:
            train_trials.append(line)

test_trials = []

with open(os.path.join(DATASET_PATH, "trials_to_test.txt")) as f:
    for line in f:
        line = line.strip()
        if line:
            test_trials.append(line)

print("Training recordings :", len(train_trials))
print("Testing recordings  :", len(test_trials))

Training recordings : 98
Testing recordings  : 24


In [ ]:
def build_metadata(trial_list):

    metadata = []

    for trial in trial_list:

        aedat_path = os.path.join(DATASET_PATH, trial)

        csv_path = os.path.join(
            DATASET_PATH,
            trial.replace(".aedat", "_labels.csv")
        )

        labels = pd.read_csv(csv_path)

        for _, row in labels.iterrows():

            metadata.append({

                "aedat": aedat_path,

                "start": int(row["startTime_usec"]),

                "end": int(row["endTime_usec"]),

                # Convert labels from 1-11 to 0-10
                "label": int(row["class"]) - 1

            })

    return metadata

In [ ]:
train_metadata = build_metadata(train_trials)

test_metadata = build_metadata(test_trials)

print("Training gestures :", len(train_metadata))
print("Testing gestures  :", len(test_metadata))

Training gestures : 1176
Testing gestures  : 288


In [ ]:
import os
import torch

PT_DATASET = "/content/drive/MyDrive/DVS128_PT"

os.makedirs(PT_DATASET, exist_ok=True)

print(PT_DATASET)

/content/drive/MyDrive/DVS128_PT


In [ ]:
metadata = []

In [ ]:
# ==========================================================
# Load an AEDAT 3.1 File
# ==========================================================

import numpy as np

def load_aedat(filename):

    with open(filename, "rb") as f:

        # Skip header
        while True:
            line = f.readline()
            if line.startswith(b"#!END-HEADER"):
                break

        packets = []

        while True:

            header = f.read(28)

            if len(header) < 28:
                break

            event_type = int.from_bytes(header[0:2], "little")
            event_size = int.from_bytes(header[4:8], "little")
            event_count = int.from_bytes(header[20:24], "little")

            data = f.read(event_size * event_count)

            if event_type != 1:
                continue

            packet = np.frombuffer(data, dtype=np.uint32)
            packet = packet.reshape(-1, 2)

            packets.append(packet)

    events = np.concatenate(packets, axis=0)

    addresses = events[:, 0]
    timestamps = events[:, 1]

    x = ((addresses >> 17) & 0x7F).astype(np.int16)
    y = ((addresses >> 2) & 0x7F).astype(np.int16)
    p = ((addresses >> 1) & 0x01).astype(np.int8)

    return x, y, p, timestamps

In [ ]:
sample_id = 0
metadata = []

for split_name, metadata_list in [
    ("train", train_metadata),
    ("test", test_metadata),
]:

    for sample in tqdm(metadata_list):

        x, y, p, t = load_aedat(sample["aedat"])

        gx, gy, gp, gt = extract_gesture(
            x, y, p, t,
            sample["start"],
            sample["end"]
        )

        voxel = events_to_voxel(gx, gy, gp, gt)

        # Save as int8 to reduce storage
        voxel = np.clip(voxel * 127, -127, 127).astype(np.int8)
        voxel = torch.from_numpy(voxel)

        filename = f"sample_{sample_id:05d}.pt"

        torch.save(
            voxel,
            os.path.join(PT_DATASET, filename)
        )

        metadata.append({
            "file": filename,
            "label": sample["label"],
            "split": split_name
        })

        sample_id += 1

print("Saved", sample_id, "samples")

  0%|          | 0/1176 [00:00<?, ?it/s]

t dtype: uint32
t min : 80048267
t max : 85092700
t0: 80048267
t1: 85092700
time_bins min: 0
time_bins max: 29


  0%|          | 1/1176 [00:02<40:00,  2.04s/it]

t dtype: uint32
t min : 89431215
t max : 95230863
t0: 89431215
t1: 95230863
time_bins min: 0
time_bins max: 29


  0%|          | 2/1176 [00:04<46:05,  2.36s/it]

...

t dtype: uint32
t min : 269521981
t max : 275622961
t0: 269521981
t1: 275622961
time_bins min: 0
time_bins max: 29


100%|█████████▉| 287/288 [06:47<00:01,  1.11s/it]

t dtype: uint32
t min : 283735754
t max : 286475098
t0: 283735754
t1: 286475098
time_bins min: 0
time_bins max: 29


100%|██████████| 288/288 [06:48<00:00,  1.42s/it]

Saved 1464 samples


In [ ]:
import pandas as pd

metadata_df = pd.DataFrame(metadata)

metadata_df.to_csv(
    os.path.join(PT_DATASET, "metadata.csv"),
    index=False
)

print(metadata_df.head())
print("Total samples:", len(metadata_df))

              file  label  split
0  sample_00000.pt      0  train
1  sample_00001.pt      1  train
2  sample_00002.pt      2  train
3  sample_00003.pt      3  train
4  sample_00004.pt      4  train
Total samples: 1464
